<a href="https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/04_day4-5_mysql.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 在浏览器中打开本 notebook 后，点击菜单 **代码执行程序 → 全部运行** 即可按顺序执行所有单元格。

# Day 4–5｜MySQL 数据库实操（两天）

**两日目标**：在你的云服务器上安装 MySQL，并掌握**建库、建表、插入、查询、修改、删除、聚合分组、多表 JOIN**，最后用 Python 连接数据库取数。这是整个学习计划的重心之一——学生管理系统的数据就存在 MySQL 里。

**两天的安排**：
- **Day 4**：安装 MySQL → 建库建表 → 增删改查（单表）
- **Day 5**：聚合统计 → 多表 JOIN → Python 连接 MySQL → **测验 1**

**操作方式**：SQL 在**服务器终端的 mysql 客户端**里执行（下面给出全部命令）；Python 连接部分先在 Colab 里用内置的 sqlite 体验，再到服务器上换成 MySQL 实战。

---
## ⏰ 今日学习节奏（每天 4 小时）

| 环节 | 时长 | 做什么 |
|---|---|---|
| 1. 讲解 | 约 1 小时 | 老师/家长带着过一遍本 notebook 的讲解部分，把知识点讲清楚 |
| 2. 自己练 | 约 2 小时 | 独立完成下面「✏️ 今日练习清单」，在 notebook 和**云服务器**上动手操作 |
| 3. 答疑 | 约 30–60 分钟 | 把练习中卡住的问题集中提出来答疑（问题随手记在文末「📒 我的问题清单」） |

> ✍️ **要求**：每一个练习都要真正动手写出来、跑出来，不要只看答案。

---
# 🗓️ Day 4：安装 + 单表增删改查

## 一、在服务器上安装 MySQL（20 分钟）

SSH 登录服务器后执行（Ubuntu）：

```bash
sudo apt update
sudo apt install -y mysql-server
sudo systemctl start mysql        # 启动
sudo systemctl enable mysql       # 开机自启
sudo systemctl status mysql       # 查看状态，看到 active (running) 即成功
```

> 💡 验证：执行 `mysql --version` 能看到版本号；`ps aux | grep mysql` 能看到进程（昨天学的命令马上用上了）。

## 二、登录 MySQL 并创建数据库（15 分钟）

```bash
sudo mysql -u root -p        # 首次直接回车或输入安装时设置的密码
```

登录成功后提示符变成 `mysql>`，**之后的 SQL 都在这里输入，每条以分号 `;` 结尾**：

```sql
-- 创建学习用数据库（字符集 utf8mb4 支持中文）
CREATE DATABASE school DEFAULT CHARACTER SET utf8mb4;

-- 创建一个学习用账号（不要用 root 做日常操作）
CREATE USER 'learn'@'%' IDENTIFIED BY 'Learn@2026';
GRANT ALL PRIVILEGES ON school.* TO 'learn'@'%';
FLUSH PRIVILEGES;

-- 查看现有数据库、切换数据库
SHOW DATABASES;
USE school;
```

> 📌 以后登录就用：`mysql -u learn -p school`（输入密码 Learn@2026）

## 三、建表 CREATE TABLE（30 分钟，重点）

常用数据类型：`INT` 整数、`VARCHAR(n)` 变长字符串、`DECIMAL(m,n)` 精确小数、`DATE` 日期。

```sql
CREATE TABLE students (
    id      INT PRIMARY KEY AUTO_INCREMENT,   -- 学号，自动递增
    name    VARCHAR(50)  NOT NULL,            -- 姓名，不允许为空
    cls     VARCHAR(50),                      -- 班级
    math    DECIMAL(5,1),                     -- 数学成绩
    english DECIMAL(5,1),                     -- 英语成绩
    enroll_date DATE                          -- 入学日期
);

SHOW TABLES;                 -- 查看有哪些表
DESCRIBE students;           -- 查看表结构（字段、类型）
```

## 四、插入数据 INSERT（20 分钟，重点）

```sql
-- 单行插入：列名和值一一对应
INSERT INTO students (name, cls, math, english, enroll_date)
VALUES ('李雷', '一班', 88, 92, '2026-02-20');

-- 一次插入多行（更高效）
INSERT INTO students (name, cls, math, english, enroll_date) VALUES
('韩梅梅', '二班', 91, 84, '2026-02-20'),
('Jim',    '一班', 76, 88, '2026-02-21'),
('Lucy',   '二班', 59, 71, '2026-02-21'),
('Lily',   '一班', 95, 60, '2026-02-22');
```

> ⚠️ 字符串和日期用**单引号**；列名顺序和值的顺序必须对应；`id` 是自增列不用写。

## 五、查询 SELECT（40 分钟，重中之重）

```sql
SELECT * FROM students;                              -- 查全部（* = 所有列）
SELECT name, math FROM students;                     -- 只查两列

SELECT * FROM students WHERE math >= 60;             -- 条件查询
SELECT * FROM students WHERE cls = '一班' AND math >= 80;   -- 多条件 AND
SELECT * FROM students WHERE math < 60 OR english < 60;     -- OR
SELECT * FROM students WHERE name LIKE '李%';        -- 模糊查询：% 匹配任意字符

SELECT * FROM students ORDER BY math DESC;           -- 按数学从高到低排序
SELECT * FROM students ORDER BY math DESC LIMIT 3;   -- 只看前 3 名
SELECT DISTINCT cls FROM students;                   -- 去重：有哪些班级
```

> 🧠 记忆口诀：**SELECT 列 FROM 表 WHERE 条件 ORDER BY 排序 LIMIT 条数**——顺序固定，不能乱。

## 六、修改 UPDATE 与删除 DELETE（20 分钟）

```sql
UPDATE students SET math = 95 WHERE name = '李雷';      -- 修改：务必带 WHERE！
UPDATE students SET math = 90, english = 90 WHERE id = 3;

DELETE FROM students WHERE id = 5;                      -- 删除：务必带 WHERE！

SELECT * FROM students;    -- 每步操作后查一下确认
```

> ⚠️ **血泪警告**：不带 WHERE 的 `UPDATE` / `DELETE` 会作用到**整张表的所有行**！先写 `SELECT ... WHERE 条件` 确认命中的是目标行，再把 SELECT 换成 UPDATE/DELETE。

---
## ✏️ Day 4 练习清单（约 2 小时，全部在服务器 mysql 客户端完成）

**A 组 · 安装与建表**
1. 完成 MySQL 安装，`systemctl status mysql` 确认运行中；用 `learn` 账号登录 `school` 库
2. 按讲义建好 `students` 表，`DESCRIBE students` 检查每个字段的类型

**B 组 · 插入与查询（家长点名要求今天掌握）**
3. 插入讲义中的 5 名学生，再自己**编造并插入 3 名**（含 1 名数学不及格的）
4. 查询全部学生；只查姓名和英语两列
5. 查出一班的所有学生；查出数学在 80~95 之间的学生（`BETWEEN 80 AND 95`）
6. 查出所有姓"李"或名字含"李"的学生（`LIKE '%李%'`）
7. 按英语成绩从高到低排序，只显示前 3 名

**C 组 · 修改与删除**
8. 把"Lucy"的数学改成 65，先用 `SELECT ... WHERE name='Lucy'` 确认再 UPDATE
9. 删除你编造的一名学生，DELETE 前同样先 SELECT 确认
10. （挑战）故意写一条不带 WHERE 的 UPDATE 会闯什么祸？**不要执行**，把后果写进问题清单旁白，答疑时讨论

---
# 🗓️ Day 5：聚合统计 + 多表 JOIN + Python 连接

## 七、聚合与分组 GROUP BY（40 分钟，重点）

```sql
SELECT COUNT(*) FROM students;                        -- 总人数
SELECT AVG(math), MAX(math), MIN(math) FROM students; -- 数学平均分/最高/最低

-- 分组统计：每个班的平均分和人数（对应 Pandas 的 groupby！）
SELECT cls, COUNT(*) AS 人数, AVG(math) AS 数学平均, AVG(english) AS 英语平均
FROM students
GROUP BY cls;

-- 分组后再筛选用 HAVING（不是 WHERE）
SELECT cls, AVG(math) AS avg_math
FROM students
GROUP BY cls
HAVING avg_math >= 80;
```

> 🧠 对照记忆：`GROUP BY` ≈ Pandas 的 `groupby()`；`HAVING` 是对**分组结果**再筛选，`WHERE` 是对**原始行**筛选。

## 八、多表 JOIN（30 分钟）

再建一张班级表，把两张表拼起来查（对应 Pandas 的 merge）：

```sql
CREATE TABLE classes (
    cls     VARCHAR(50) PRIMARY KEY,
    teacher VARCHAR(50),
    room    VARCHAR(20)
);
INSERT INTO classes VALUES ('一班', '张老师', '301'), ('二班', '王老师', '302');

-- JOIN：把学生和班主任信息拼在一起
SELECT s.name, s.cls, s.math, c.teacher, c.room
FROM students s
JOIN classes c ON s.cls = c.cls;
```

## 九、Python 连接数据库（30 分钟）

先在 Colab 里用 sqlite 体验"Python ↔ 数据库"的完整链路（语法和 MySQL 几乎一样）：

In [ ]:
import sqlite3
import pandas as pd

# 1. 连接数据库（sqlite 版：文件就是数据库）
conn = sqlite3.connect("school_demo.db")

# 2. 用 pandas 执行建表和插入（实际工作中用 SQLAlchemy/客户端工具）
conn.executescript("""
DROP TABLE IF EXISTS students;
CREATE TABLE students (id INTEGER PRIMARY KEY AUTOINCREMENT,
                       name TEXT, cls TEXT, math REAL, english REAL);
INSERT INTO students (name, cls, math, english) VALUES
('李雷','一班',88,92),('韩梅梅','二班',91,84),('Jim','一班',76,88),
('Lucy','二班',59,71),('Lily','一班',95,60);
""")
conn.commit()

# 3. read_sql：一句 SQL 直接得到 DataFrame —— 数据库和 Pandas 的桥梁
df = pd.read_sql("SELECT * FROM students WHERE math >= 60", conn)
print(df)

# 4. 分组统计也可以直接拿回来
print(pd.read_sql("SELECT cls, AVG(math) AS avg_math FROM students GROUP BY cls", conn))
conn.close()

**在云服务器上连接真正的 MySQL**（Day 5 练习里实操）：

```bash
# 服务器上先允许远程连接（给 Colab/自己电脑连进来用）：
sudo sed -i 's/bind-address.*/bind-address = 0.0.0.0/' /etc/mysql/mysql.conf.d/mysqld.cnf
sudo systemctl restart mysql
# 并在云服务商控制台的安全组里放行 3306 端口（仅建议学习期间开放）
```

```python
# Colab 里执行（换成你的服务器 IP 和密码）：
!pip install pymysql -q
import pymysql, pandas as pd
conn = pymysql.connect(host="你的服务器IP", user="learn",
                       password="Learn@2026", database="school", charset="utf8mb4")
df = pd.read_sql("SELECT * FROM students", conn)
conn.close()
```

> 🔐 安全提醒：3306 端口只对学习需要临时开放，密码不要写进会公开分享的 notebook。

---
## ✏️ Day 5 练习清单（约 2 小时）

**A 组 · 聚合（服务器 mysql 客户端）**
1. 统计 students 表总人数、全班数学平均分（保留 1 位小数：`ROUND(AVG(math),1)`）
2. 按班级分组统计人数和两科平均分，按数学平均分从高到低排序
3. 用 HAVING 找出"英语平均分 ≥ 80"的班级
4. （挑战）统计每个班的**及格率**：`SELECT cls, SUM(math>=60)/COUNT(*) AS 及格率 FROM students GROUP BY cls;`（`math>=60` 结果是 1/0，求和即及格人数）

**B 组 · JOIN**
5. 建好 classes 表并完成讲义中的 JOIN 查询
6. （挑战）查询"王老师"班上的所有学生姓名和成绩

**C 组 · Python 连接（重头戏）**
7. 在服务器上完成远程访问配置，在 Colab 里用 pymysql 连上**你自己的 MySQL**，`read_sql` 查出 students 全表
8. 用 Pandas 对查回来的数据算各班平均分，和你在 mysql 客户端里 GROUP BY 的结果**核对一致**
9. 用 Python 向 MySQL 插入一名新学生，然后在 mysql 客户端 `SELECT` 确认（提示：`cursor.execute(INSERT...)` + `conn.commit()`）

---
## 📝 完成今日学习后：去做【测验 1】

测验 1 覆盖 Day 1–5 全部内容（Python 字符串、Pandas、Linux、MySQL），在学习计划网页版中打开。**80 分以上再进入项目阶段**，不够的章节回到对应 notebook 补漏。

---
## 📒 我的问题清单（随手记录，答疑前整理）

学习中遇到任何卡住的地方，立刻记在下面表格里，**答疑时间集中解决**：

| 日期 | 问题描述 | 状态（待问/已解决） | 答案要点 |
|---|---|---|---|
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |

---
## ✅ 参考答案（先自己做，再对照）

```sql
-- Day4 A2
DESCRIBE students;

-- Day4 B5
SELECT * FROM students WHERE math BETWEEN 80 AND 95;
-- Day4 B6
SELECT * FROM students WHERE name LIKE '%李%';
-- Day4 B7
SELECT * FROM students ORDER BY english DESC LIMIT 3;
-- Day4 C8
SELECT * FROM students WHERE name = 'Lucy';
UPDATE students SET math = 65 WHERE name = 'Lucy';

-- Day5 A1
SELECT COUNT(*), ROUND(AVG(math),1) FROM students;
-- Day5 A2
SELECT cls, COUNT(*) AS 人数, AVG(math) AS 数学平均, AVG(english) AS 英语平均
FROM students GROUP BY cls ORDER BY 数学平均 DESC;
-- Day5 A3
SELECT cls, AVG(english) AS avg_en FROM students GROUP BY cls HAVING avg_en >= 80;
-- Day5 B6
SELECT s.name, s.math, s.english FROM students s
JOIN classes c ON s.cls = c.cls WHERE c.teacher = '王老师';
```

```python
# Day5 C9 插入示例
cursor = conn.cursor()
cursor.execute("INSERT INTO students (name, cls, math, english) VALUES (%s,%s,%s,%s)",
               ("测试同学", "三班", 77, 88))
conn.commit()
```

---
➡️ **接下来**：Day 6–15 学生管理系统项目 —— 打开 [https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/05_day6-15_project_tasks.ipynb](https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/05_day6-15_project_tasks.ipynb)